# Genes

This notebook shows embpy's gene workflow across model families: DNA sequence models, protein-language models, text models, static lookup embeddings, morphology-backed perturbation embeddings, annotation utilities, and embedding comparison plots.

Genes can be treated as feature-level biological entities or as row-aligned perturbation/action labels. embpy keeps those two cases explicit so downstream AnnData objects remain easy to reason about.


## embpy capability map

All embpy tutorials follow the same package pattern:

1. `BioEmbedder.embed(...)` is the single entry point for genes, proteins, molecules, perturbation images, text, and single cells.
2. Resolvers convert biological identifiers into model-ready inputs, such as gene sequences, protein sequences, SMILES strings, microscopy tensors, or AnnData matrices.
3. The model registry selects the requested embedding backend, from lightweight local baselines to foundation models.
4. Preprocessing utilities in `embpy.pp` prepare inputs when a model needs a specific representation, such as raw counts, log-normalized expression, or morphology canvases.
5. Metadata tools in `embpy.tl` and `embpy.resources` annotate the resulting AnnData with genes, proteins, molecules, perturbations, and cell-line metadata.
6. Plotting and comparison helpers in `embpy.pl` and `embpy.tl` inspect embedding geometry, cluster structure, KNN overlap, similarity, and annotation enrichment.

The important contract is that embeddings are stored in AnnData-friendly locations: row-aligned embeddings in `.obsm`, feature-aligned embeddings in `.varm`, and entity-aligned payloads or provenance in `.uns`. `.X` stays reserved for count/expression-like data or a lightweight placeholder.


## What this notebook demonstrates

- Gene identifiers are resolved from symbols or Ensembl IDs into model-ready DNA, protein, text, or static embedding inputs.
- Feature-like gene embeddings are stored in `.varm`, while perturbation/action embeddings are stored in `.obsm` or `.uns` depending on alignment.
- Multiple model families can be compared with KNN overlap, cross-embedding correlation, norms, and annotated scatter plots.
- Gene and protein metadata can be joined back onto the same AnnData object for interpretation.


In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display

from embpy import BioEmbedder, pl, tl

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

embedder = BioEmbedder(device="auto", organism="human")


def compact_obs(adata: ad.AnnData, prefixes: tuple[str, ...], base: list[str] | None = None) -> pd.DataFrame:
    base = base or []
    cols = [c for c in base if c in adata.obs.columns]
    cols += [c for c in adata.obs.columns if c.startswith(prefixes)]
    return adata.obs.loc[:, list(dict.fromkeys(cols))]


genes = ["TP53", "EGFR", "MYC", "BRCA1", "JUN", "STAT1", "IRF1", "CDK1"]
print(f"Genes: {genes}")

gene_feature_adata = ad.AnnData(
    X=np.zeros((1, len(genes)), dtype=np.float32),
    obs=pd.DataFrame(index=["example_cell"]),
    var=pd.DataFrame({"gene_symbol": genes}, index=pd.Index(genes, name="gene_symbol")),
)

gene_perturbation_adata = ad.AnnData(
    X=np.zeros((len(genes), 1), dtype=np.float32),
    obs=pd.DataFrame(
        {"perturbation": genes, "symbol": genes},
        index=pd.Index(genes, name="perturbation"),
    ),
    var=pd.DataFrame(index=["placeholder_feature"]),
)

display(gene_feature_adata)
display(gene_perturbation_adata)


## Embed genes with DNA, protein, and text models

Gene embeddings can describe gene features or gene perturbation labels. The first call stores a feature-aligned DNA embedding in `.varm`. The second workflow uses an AnnData whose rows are perturbation labels and sets `is_perturbation=True`, so DNA, protein, and text gene embeddings land directly in `.obsm`. For the DNA demo we embed exon sequences with a 32k-context HyenaDNA model; full gene loci are often much longer than 1k-context models can handle comfortably.


In [ ]:
gene_feature_adata = embedder.embed(
    gene_feature_adata,
    entity_type="gene",
    id_type="symbol",
    var_column="gene_symbol",
    model="hyenadna_small_32k",
    output="anndata",
    key="X_gene_dna",
    organism="human",
    region="exons",
    pooling_strategy="mean",
    show_progress=True,
)

print("feature-aligned varm keys:", list(gene_feature_adata.varm.keys()))

gene_space = gene_perturbation_adata.copy()
gene_models = [
    ("hyenadna_small_32k", "X_gene_dna", {"region": "exons"}),
    ("esm2_8M", "X_gene_protein", {}),
    ("minilm_l6_v2", "X_gene_text", {}),
]
for model_name, key, model_kwargs in gene_models:
    gene_space = embedder.embed(
        gene_space,
        entity_type="gene",
        id_type="symbol",
        obs_column="perturbation",
        model=model_name,
        output="anndata",
        is_perturbation=True,
        key=key,
        organism="human",
        pooling_strategy="mean",
        show_progress=True,
        **model_kwargs,
    )

display(gene_space)
print("perturbation-aligned obsm keys:", list(gene_space.obsm.keys()))
print("embedding metadata keys:", list(gene_space.uns["embeddings"].keys()))


In [ ]:
print("Embeddings metadata:", gene_space.uns["perturbations"])

## Add morphology perturbation embeddings

Perturbation/action embeddings are entity-aligned payloads in `.uns`. When a downstream model needs one vector per row, materialize the payload into `.obsm` explicitly.



In [ ]:
from embpy.io import materialize_perturbation_obsm

perturbation_adata = ad.AnnData(
    X=np.zeros((len(genes), 1), dtype=np.float32),
    obs=pd.DataFrame({"perturbation": genes}, index=[f"pert_{i}" for i in range(len(genes))]),
)

perturbation_adata = embedder.embed(
    perturbation_adata,
    entity_type="perturbation",
    obs_column="perturbation",
    model="subcell_mae_rybg",
    output="anndata",
    attach_to="uns",
    key="X_pert_subcell_mae_rybg",
    morphology_dataset="hpa",
    morphology_source="subcell",
    max_images=1,
    aggregation="mean",
    show_progress=True,
)
perturbation_adata = materialize_perturbation_obsm(
    perturbation_adata,
    embedding_key="X_pert_subcell_mae_rybg",
    perturbation_key="perturbation",
    obsm_key="X_pert_subcell_mae_rybg",
    missing="nan",
)

print("uns perturbation keys:", list(perturbation_adata.uns["perturbations"].keys()))
print("materialized shape:", perturbation_adata.obsm["X_pert_subcell_mae_rybg"].shape)



## Annotate genes and their protein products

Annotations are fetched through embpy's real metadata utilities and stored in `.obs` and `.uns`.



In [ ]:
gene_space = tl.annotate_gene_perturbations(
    gene_space,
    column="symbol",
    sources=["pathways", "interactions", "diseases"],
    copy=True,
)
gene_space = tl.annotate_proteins(
    gene_space,
    column="symbol",
    id_type="symbol",
    sources=["metadata", "location", "domains", "go", "interactions"],
    copy=True,
)

display(compact_obs(gene_space, ("gene_", "prot_"), base=["symbol", "perturbation", "gene_symbol"]))
print("annotation stores:", [k for k in gene_space.uns if "annotation" in k])


## Plot annotated embedding spaces



In [ ]:
color_key = "gene_n_pathways" if "gene_n_pathways" in gene_space.obs else "symbol"
pl.plot_embedding_space(
    gene_space,
    obsm_key="X_gene_dna",
    method="pca",
    color=color_key,
    annotate=True,
    annotate_col="symbol",
    title="DNA gene perturbation embeddings colored by embpy gene annotations",
)

if "prot_location" in gene_space.obs:
    pl.plot_embedding_space(
        gene_space,
        obsm_key="X_gene_protein",
        method="pca",
        color="prot_location",
        annotate=True,
        annotate_col="symbol",
        title="Protein-derived gene perturbation embeddings colored by UniProt location",
    )


## Compare embedding views



In [ ]:
k = min(3, gene_space.n_obs - 1)
_, mean_overlap = tl.compute_knn_overlap(gene_space, "X_gene_dna", "X_gene_protein", k=k)
print(f"Mean DNA/protein KNN overlap: {mean_overlap:.3f}")

pl.knn_overlap(gene_space, obsm_keys=["X_gene_dna", "X_gene_protein", "X_gene_text"], k=k)
pl.cross_embedding_correlation(gene_space, "X_gene_dna", "X_gene_protein")
pl.embedding_norms(gene_space, obsm_keys=["X_gene_dna", "X_gene_protein", "X_gene_text"])


## Summary

This notebook covered the gene-level embpy pattern: resolve gene identifiers, generate embeddings from several foundation-model families, attach them to the correct AnnData axis, enrich the object with biological metadata, compare embedding geometries, and save reusable AnnData artifacts.


## Save a reusable artifact



In [ ]:
gene_feature_adata.write_h5ad(OUTPUT_DIR / "gene_feature_embeddings.h5ad")
gene_space.write_h5ad(OUTPUT_DIR / "gene_perturbation_embeddings.h5ad")
print(OUTPUT_DIR / "gene_feature_embeddings.h5ad")
print(OUTPUT_DIR / "gene_perturbation_embeddings.h5ad")
